In [ ]:
# Transformers example -> Sorting a sequence of numbers


The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


In [2]:
import flax.nnx as nnx
import jax
import jax.numpy as jnp
import jax.random as jrandom
import optax

from probjax.nn import PosEncode, Transformer, RotaryPosEncode, MultiHeadAttention, LearnablePosEncode
from probjax.nn.layers.attention import flex_attention
from probjax.nn.pallas_kernels.attention_mask_bias import SeqLenMask, SymmetricAlibiBias

In [3]:
VOCAB_SIZE = 10

In [4]:
import jax.numpy as jnp
import jax.random as jrandom


def generate_data(key, n, T, vocab_size=10, lengths=None):
    if lengths is None:
        lengths = jnp.array(T) * jnp.ones(n, jnp.int32)

    m   = jnp.arange(T)[None, :] < lengths[:, None]                 # (n,T)
    tok = jrandom.randint(key, (n, T), 0, vocab_size, jnp.int32)  # (n,T)
    seq = jnp.where(m, tok, 9).astype(jnp.int32)               # (n,T)
    label = jnp.sort(seq, axis=-1)                             # (n,T)

    return seq[..., None], label[..., None]                    # (n,T,1),(n,T,1),(n,),(n,T)                # (n,T,1), (n,T,1), (n,T)

inputs, labels= generate_data(jax.random.PRNGKey(0), 1000, 10, VOCAB_SIZE)

In [5]:
inputs, labels= generate_data(jax.random.PRNGKey(0), 1000, 10, VOCAB_SIZE, lengths=jnp.array(2)*jnp.ones(1000, jnp.int32))

In [6]:
key = jrandom.PRNGKey(0)

In [7]:
from functools import partial

class Model(nnx.Module):

    def __init__(self, dim,rngs, dropout_rate=0.):
        self.embed = nnx.Embed(VOCAB_SIZE, dim, rngs=rngs)
        self.pos_embed = RotaryPosEncode(dim, max_seq_len=4096, rngs=rngs)
        self.transformer = Transformer(dim, 8,8,8, widening_factor=4,rngs=rngs, dropout_rate=dropout_rate, attention_fn=flex_attention, norm_cls=partial(nnx.GroupNorm, num_groups=32))
        self.output = nnx.Linear(dim, VOCAB_SIZE, rngs=rngs)

    def __call__(self, x, deterministic=False, mask=None, bias=None):
        bias = SymmetricAlibiBias()
        x = self.embed(x)
        x = jnp.squeeze(x,axis=-2)
        x = self.pos_embed(x)
        x = self.transformer(x,deterministic=deterministic, mask=mask, bias=bias)
        x = self.output(x)
        return x


In [8]:
model = Model(256, rngs=nnx.Rngs(0))

In [9]:
nnx.display(model)

In [10]:
graphdef, params, state = nnx.split(model, nnx.Param,...)

In [11]:
optimizer = optax.chain(optax.adaptive_grad_clip(2.), optax.radam(1e-4))
opt_state = optimizer.init(params)

In [12]:
max_train_seqs = [128, 2048]
batch_size = 128
def loss_fn_seq_len(params, state,key):
    model = nnx.merge(graphdef, params, state, copy=True)
    model.train()
    losses = []
    for max_train_seq in max_train_seqs:
        key, key_sub, k_len = jax.random.split(key, 3)
        # sample lengths biased towards smaller values
        u = jax.random.uniform(k_len, (batch_size,), minval=0.0, maxval=1.0)
        alpha = 3.0  # larger alpha => stronger bias to small lengths
        lengths = ((u ** alpha) * (max_train_seq - 1)).astype(jnp.int32) + 1
        inp_data, labels = generate_data(key_sub, batch_size, max_train_seq, lengths=lengths, vocab_size=VOCAB_SIZE)

        mask = SeqLenMask(lengths)#.dense(max_train_seq, max_train_seq, batch_size=batch_size)
        logits = model(inp_data, mask=mask)

        # xent per token
        labels_oh = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)  # (B,S,V)
        token_loss = optax.softmax_cross_entropy(logits, labels_oh)      # (B,S)

        # mask out padded positions and average per *token*
        length_mask = jnp.arange(max_train_seq)[None, :] < lengths[:, None]          # (B,S)
        token_loss = jnp.where(length_mask, token_loss, 0.0)
        loss = token_loss.sum() / length_mask.sum()
        losses.append(loss)
    loss = sum(losses) / len(losses)
    _, _, state = nnx.split(model, nnx.Param,...)
    return loss, state


@jax.jit
def acc(params, state, inputs, outputs):
    model = nnx.merge(graphdef, params, state)
    model.eval()
    inp_data, labels = inputs, outputs
    logits = model(inp_data)
    labels = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)
    acc = (logits.argmax(axis=-1) == labels.argmax(-1)).mean()
    return acc

@jax.jit
def update(params, state, key, opt_state):
    (loss, state), grads = jax.value_and_grad(loss_fn_seq_len, has_aux=True)(params, state, key)
    updates, opt_state = optimizer.update(grads, opt_state, params=params)
    params = optax.apply_updates(params, updates)
    return loss, params, state, opt_state

In [13]:
key = jrandom.PRNGKey(0)

In [14]:

for i in range(5000):
    key, subkey, key2 = jrandom.split(key, 3)
    loss, params, state, opt_state = update(params, state, key, opt_state)
    if (i % 500) == 0:
        inputs, labels = generate_data(key, 512, 128, vocab_size=VOCAB_SIZE)
        accuracy = acc(params, state, inputs, labels)
        print(accuracy, loss)

0.10418701 2.3635225
0.87713623 0.29636654
0.9187317 0.1500524
0.94317627 0.12039995
0.9513855 0.17029172
0.9654846 0.07211572
0.97058105 0.059484985


KeyboardInterrupt: 

In [15]:
model.eval()
nnx.update(model, params)

In [17]:
input = jax.random.randint(key+5, (1, 32,1),0, 10,dtype=jnp.int32)
outputs = model(input)
print(input[0,...,0])
print(outputs.argmax(-1)[0])
print(jnp.sort(input[0,...,0]))
print(jnp.all(outputs.argmax(-1)[0] == jnp.sort(input[0,...,0])))

[7 8 2 0 4 9 5 5 2 8 2 2 8 9 6 5 7 0 7 9 9 8 7 1 7 1 5 4 9 8 7 6]
[0 0 0 0 1 1 1 2 2 2 2 5 5 5 5 6 7 7 7 7 7 7 7 7 7 7 7 7 7 7 7 7]
[0 0 1 1 2 2 2 2 4 4 5 5 5 5 6 6 7 7 7 7 7 7 8 8 8 8 8 9 9 9 9 9]
False


In [ ]:
jnp.allclose(outputs.argmax(-1)[0], jnp.sort(input[0,...,0]))

Array(False, dtype=bool)